# FoldPipe: Optimized Training on Kaggle

This notebook demonstrates the optimized pipeline for Molecular Dynamics simulation using `TorchMD-Net` on the MD17 dataset. We apply hardware-aware optimizations (mixed precision, prefetching, parallel workers) to maximize throughput.

In [ ]:
!pip install torchmd-net

In [ ]:
import torch
import time
from torch_geometric.loader import DataLoader
from torchmdnet.datasets import MD17

# PyTorch 2.6 defaults torch.load to weights_only=True, which breaks torchmd-net cache unpickling.
_original_load = torch.load
torch.load = lambda *args, **kwargs: _original_load(*args, **{**kwargs, 'weights_only': False})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def get_optimized_dataloader(data_dir='./md17_data', molecules='aspirin', batch_size=128, num_workers=2):
    dataset = MD17(data_dir, molecules=molecules)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,          # Accelerates CPU to GPU data transfer
        prefetch_factor=2,        # Pre-fetch 2 batches per worker
        persistent_workers=True   # Keep worker processes alive between epochs
    )
    return dataloader

In [ ]:
def train_optimized(epochs=2):
    print(f"Starting optimized training on {device}")
    dataloader = get_optimized_dataloader(batch_size=128, num_workers=2) # Adjust num_workers for Kaggle
    
    # Mock model for profiling
    model = torch.nn.Linear(100, 100).to(device) 
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Mixed Precision Setup
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    
    for epoch in range(epochs):
        model.train()
        start_time = time.time()
        
        for i, batch in enumerate(dataloader):
            try:
                z = batch.z.to(device, non_blocking=True)
                pos = batch.pos.to(device, non_blocking=True)
                y = batch.y.to(device, non_blocking=True)
            except AttributeError:
                pass
            
            optimizer.zero_grad(set_to_none=True) # Slightly faster than zero_grad()
            
            # Forward pass with AMP
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                mock_input = torch.randn(128, 100, device=device)
                loss = model(mock_input).sum()
                
            # Backward pass with scaler
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            # Memory Management: aggressive cache clearing
            if i > 0 and i % 50 == 0:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                print(f"Batch {i} processed, elapsed epoch time: {time.time() - start_time:.2f} seconds")
                
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch} completed in {epoch_time:.2f} seconds.")

train_optimized(epochs=2)